# Implementação do Agente Laya: descarte de regras obsoletas por falta de relevância
O núcleo do RAG Agêntico é desacoplar a recuperação (banco de dados) do julgamento sobre *o que deve chegar ao LLM*.

No **Notebook 03**, criamos um fluxo de duas etapas: primeiro o agente Laya atua como **juiz** de cada trecho recuperado e só depois o Llama 3.1 atua como **gerador**.

**Regra de relevância:** um trecho é irrelevante quando existe, entre os resultados, uma versão **mais recente da mesma regra**. A Laya responde à pergunta difícil ("qual regra este trecho define?") e a data de importação desempata.

## Por que a Laya classifica, em vez de dar uma nota de relevância?
O `laya:322m` é um modelo de **decisão** (ModernBERT, capacidade `decision` no Ollama), não um LLM gerativo. A API `ollama.systemone` oferece três tipos de pergunta: `noul` (probabilidade de verdadeiro), `choice` (escolher uma opção) e `score` (escala ordenada).

Na primeira versão deste projeto usamos `noul` perguntando "este texto é relevante para a pergunta do cliente?". O resultado foi **0.99–1.00 para todos os 5 chunks**, inclusive os de Brinquedos e Móveis, e o filtro aprovou 5 de 5. Testamos outras formulações (pergunta e trecho no mesmo `state`, `score` de utilidade, `choice` de relação), e em todas o menor score dos trechos relevantes ficou abaixo do maior score dos irrelevantes: **nenhum threshold separa os dois grupos**. Para um classificador desse porte, qualquer trecho de política de devolução "parece responder" a uma pergunta sobre devolução.

O que a Laya faz muito bem é **classificar um único texto** em categorias. Ao perguntar "a qual categoria de produto esta regra se aplica?", ela acerta todos os chunks da base. Nas regras de produto a confiança fica entre 0.76 e 1.00; nas regras gerais ela acerta com confiança menor, e por isso a classe `geral` nunca é usada para descartar nada. Por isso usamos `choice`.

## 1. Conexão Base (Setup)
Igual ao Notebook 02, reconectamos aos serviços.

In [1]:
import ollama
from qdrant_client import QdrantClient

EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
LLM_MODEL = "llama3.1"
LAYA_MODEL = "laya:322m-multilingual-bf16"
TOP_K = 5

# Abaixo deste nível de confiança a Laya não tem certeza de qual regra o trecho define.
# Nesse caso o trecho não descarta nem é descartado (fail-open): melhor sobrar contexto do que perder uma regra vigente.
CONFIANCA_MINIMA = 0.5

In [2]:
client = QdrantClient(url="http://localhost:6333")
collection_name = "portfolio_agentic_rag"

## 2. Criação do Agente de Relevância (Laya)
As descrições das categorias importam: os exemplos de produtos ajudam o modelo e a classe `geral` absorve as regras que valem para todas as categorias (nota fiscal, análise técnica, reembolso). Regras gerais nunca são descartadas por versão, porque cobrem assuntos diferentes entre si.

In [3]:
CATEGORIAS = {
    "roupas": "Roupas, calçados e vestuário",
    "moveis": "Móveis e decoração (sofás, mesas, cadeiras)",
    "brinquedos": "Brinquedos e artigos infantis",
    "eletronicos": "Eletrônicos e informática (laptops, notebooks, celulares, tablets)",
    "linha_branca": "Eletrodomésticos da linha branca (geladeira, fogão, lavadora)",
    "esportivos": "Artigos esportivos e suplementos (whey protein, creatina)",
    "geral": "Regras gerais que valem para todas as categorias (nota fiscal, análise técnica, reembolso)",
}

def classificar_regra(texto: str) -> tuple[str, float]:
    """Usa a Laya para descobrir a qual categoria de produto um trecho da política se aplica."""
    if not texto:
        raise ValueError("O texto do trecho não pode ser vazio.")

    response = ollama.systemone(
        model=LAYA_MODEL,
        state=texto,
        questions={
            "categoria": {
                "type": "choice",
                "instructions": "A qual categoria de produto esta regra se aplica?",
                "criteria": CATEGORIAS,
            },
        },
    )
    resposta = response.answers["categoria"]
    return resposta.choice, resposta.confidence

## 3. Triagem: descartar versões obsoletas
Para cada trecho recuperado, a Laya define a categoria. Em seguida, dentro de cada categoria, **só a importação mais recente é relevante**. As anteriores recebem relevância zero e são podadas.

A atualização de 2024 foi parcial, então um simples filtro por data (ex.: "só documentos de 2024") apagaria regras de 2021 que continuam vigentes, como Móveis e Roupas. É a classificação da Laya que permite descartar **apenas** o que foi substituído.

In [4]:
def filtrar_relevantes(resultados, verbose: bool = True):
    avaliados = []
    for res in resultados:
        categoria, confianca = classificar_regra(res.payload["texto"])
        avaliados.append((res, categoria, confianca))

    # Data da versão mais recente de cada regra, considerando só classificações confiáveis
    versao_vigente = {}
    for res, categoria, confianca in avaliados:
        if categoria != "geral" and confianca >= CONFIANCA_MINIMA:
            data = res.payload["data_importacao"]
            versao_vigente[categoria] = max(versao_vigente.get(categoria, data), data)

    aprovados = []
    for res, categoria, confianca in avaliados:
        data = res.payload["data_importacao"]
        obsoleto = (
            categoria in versao_vigente
            and confianca >= CONFIANCA_MINIMA
            and data < versao_vigente[categoria]
        )
        if not obsoleto:
            aprovados.append(res)
        if verbose:
            decisao = f"DESCARTADO (substituído pela versão de {versao_vigente[categoria]})" if obsoleto else "APROVADO"
            print(f"[{data}] {categoria:<13} conf={confianca:.2f} -> {decisao}")
            print(f"    {res.payload['texto'][:80]}...")
    return aprovados

## 4. Execução: mesma pergunta do Notebook 02

In [5]:
pergunta = "Comprei um laptop ontem, abri a caixa para olhar e me arrependi. Qual o prazo e a regra para devolver?"

def buscar(pergunta: str):
    vetor = ollama.embed(model=EMBEDDING_MODEL, input=f"search_query: {pergunta}")["embeddings"][0]
    return client.query_points(collection_name=collection_name, query=vetor, limit=TOP_K).points

resultados = buscar(pergunta)

In [6]:
print("--- INICIANDO TRIAGEM PELO AGENTE LAYA ---\n")
documentos_filtrados = filtrar_relevantes(resultados)
print(f"\nResumo da Triagem: {len(resultados)} brutos do Qdrant -> {len(documentos_filtrados)} aprovados pela Laya.")

--- INICIANDO TRIAGEM PELO AGENTE LAYA ---



[2021-03-10] eletronicos   conf=1.00 -> DESCARTADO (substituído pela versão de 2024-03-01)
    5. Regras para Eletrônicos e Informática (Laptops, Notebooks, Celulares, Tablets...
[2024-03-01] eletronicos   conf=0.78 -> APROVADO
    5. Regras para Eletrônicos e Informática (Laptops, Notebooks, Celulares, Tablets...
[2021-03-10] brinquedos    conf=1.00 -> APROVADO
    4. Regras para Brinquedos e Artigos Infantis
Brinquedos possuem prazo de arrepen...
[2021-03-10] moveis        conf=1.00 -> APROVADO
    3. Regras para Móveis e Decoração
O prazo de arrependimento para móveis (sofás, ...
[2021-03-10] geral         conf=0.22 -> APROVADO
    8. Análise Técnica e Reembolso
O produto devolvido passa por análise técnica no ...

Resumo da Triagem: 5 brutos do Qdrant -> 4 aprovados pela Laya.


In [7]:
# Mesmo prompt do Notebook 02: a única variável do experimento é o filtro
PROMPT_SISTEMA = """Você é um assistente de atendimento ao cliente da Loja Demo.
Responda à pergunta do usuário baseando-se APENAS no contexto fornecido.

CONTEXTO:
{contexto}
"""

def gerar_resposta(pergunta: str, documentos) -> str:
    if not documentos:
        return "[FLUXO ALTERNATIVO]: Nenhum documento relevante. Encaminhar para atendimento humano."
    contexto = "\n\n".join(doc.payload["texto"] for doc in documentos)
    resposta = ollama.chat(
        model=LLM_MODEL,
        options={"temperature": 0},
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA.format(contexto=contexto)},
            {"role": "user", "content": pergunta}
        ]
    )
    return resposta["message"]["content"]

print("--- RESPOSTA FINAL DO SISTEMA (LLAMA 3.1) ---")
print(gerar_resposta(pergunta, documentos_filtrados))

--- RESPOSTA FINAL DO SISTEMA (LLAMA 3.1) ---


Você pode devolver o laptop em até 30 dias corridos após o recebimento. Como a caixa foi aberta, isso não é problema. Basta enviar o produto junto com todos os acessórios, cabos e manuais originais. O aparelho não pode ter danos físicos.


## 5. Comparação em lote: Naive RAG vs Agentic RAG
Uma única pergunta não prova nada. Rodamos o mesmo pipeline para casos afetados pela atualização de 2024 e para um **caso de controle** (sofá), cuja regra não mudou. No caso de controle o filtro não pode atrapalhar.

In [8]:
CASOS = [
    ("Laptop com caixa aberta", "Comprei um laptop ontem, abri a caixa para olhar e me arrependi. Qual o prazo e a regra para devolver?"),
    ("Notebook já ligado", "Comprei um notebook há duas semanas, já liguei e não gostei. Consigo devolver?"),
    ("Creatina com pote aberto", "Comprei creatina há 12 dias, abri o pote e usei só um pouco. Posso devolver?"),
    ("Controle: sofá", "Comprei um sofá e não combinou com a sala, posso devolver?"),
]

for titulo, pergunta_caso in CASOS:
    brutos = buscar(pergunta_caso)
    filtrados = filtrar_relevantes(brutos, verbose=False)
    print("=" * 100)
    print(f"{titulo.upper()} | {len(brutos)} brutos -> {len(filtrados)} aprovados")
    print(f"\n[NAIVE]   {gerar_resposta(pergunta_caso, brutos)}")
    print(f"\n[AGENTIC] {gerar_resposta(pergunta_caso, filtrados)}\n")

LAPTOP COM CAIXA ABERTA | 5 brutos -> 4 aprovados



[NAIVE]   Lamento ouvir que você se arrependeu da compra!

De acordo com as nossas regras, se você abriu a caixa do laptop para olhar, mesmo que apenas para conferência visual, você perde o direito à devolução por arrependimento.

No entanto, você pode considerar a opção de devolução por defeito, desde que o laptop apresente algum problema técnico. Nesse caso, você precisará enviar o produto para análise técnica no Centro de Distribuição e, se a devolução for aprovada, você receberá o reembolso de acordo com as nossas regras.

Se você tiver alguma dúvida ou precisar de mais informações, por favor, entre em contato conosco. Estamos aqui para ajudar!



[AGENTIC] Você pode devolver o laptop em até 30 dias corridos após o recebimento. Como a caixa foi aberta, isso não é problema. Basta enviar o produto junto com todos os acessórios, cabos e manuais originais. O aparelho não pode ter danos físicos.



NOTEBOOK JÁ LIGADO | 5 brutos -> 4 aprovados



[NAIVE]   Peço desculpas, mas infelizmente não é possível devolver o notebook por arrependimento. De acordo com nossas regras, o prazo para devolução de eletrônicos por arrependimento é de 7 (sete) dias corridos após o recebimento. Como você comprou o notebook há duas semanas, o prazo já foi ultrapassado. No entanto, você pode entrar em contato conosco para saber mais sobre a garantia do fabricante e se o seu produto apresenta algum problema.


...


[NAIVE]   Infelizmente, não. De acordo com as regras da Loja Demo, os suplementos alimentares têm prazo de 7 (sete) dias para devolução, e o lacre de alumínio interno do pote deve estar inviolado. Como você abriu o pote e usou um pouco da creatina, não podemos aceitar a devolução.



[AGENTIC] Sim, você pode devolver a creatina. Como você usou apenas um pouco do conteúdo e o pote ainda está aberto, desde que pelo menos metade do conteúdo não tenha sido consumida, você pode devolver o produto em até 15 dias.



CONTROLE: SOFÁ | 5 brutos -> 5 aprovados



[NAIVE]   Sim, você pode devolver o sofá. O prazo de arrependimento para móveis é de 15 dias corridos e, como você abriu a embalagem para avaliar o sofá, a devolução é permitida. Basta agendar a coleta com a transportadora especializada e apresentar a nota fiscal eletrônica (DANFE) enviada por e-mail.



[AGENTIC] Sim, você pode devolver o sofá. O prazo de arrependimento para móveis é de 15 dias corridos e, como você abriu a embalagem para avaliar o sofá, a devolução é permitida. Basta agendar a coleta com a transportadora especializada e apresentar a nota fiscal eletrônica (DANFE) enviada por e-mail.



---
### Conclusão e Impacto Arquitetural

1. **Mitigação de Alucinações por conteúdo obsoleto:** o LLM gerador nunca vê duas versões da mesma regra, então não precisa adivinhar qual é a vigente.
2. **Cada modelo faz o que sabe fazer:** a Laya (322M) classifica, a data de importação desempata e o Llama 3.1 (8B) só escreve a resposta. O julgamento não é delegado ao modelo grande.
3. **Precisão cirúrgica:** como a atualização foi parcial, regras antigas que continuam valendo (Móveis, Roupas, Nota Fiscal) seguem no contexto. O caso de controle mostra isso.
4. **Lição de engenharia:** perguntar "isto é relevante?" a um classificador pequeno não funcionou (tudo deu ~1.0). Reformular o problema como uma classificação de texto único tornou o filtro confiável.